參考資料 :
1.  https://blog.jiatool.com/posts/pchome_spider01/ Pchome上
2. https://blog.jiatool.com/posts/pchome_spider02/  Pchome下
3. https://ithelp.ithome.com.tw/articles/10209356 User-Agent 說明與使用

In [6]:
from IPython.core.payloadpage import page
! pip install pandas requests


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import json
from pathlib import Path
import random
import re
import time
import pandas as pd
import requests

params  :
 1. 用來設定「網址查詢參數」的字典參數
 2. 自動把參數拼接到網址後面
3. 遇到中文（如 q="台北美食"）或特殊符號時，requests 會自動幫你做 URL 編碼

# params = {Key : Value}
### Key ( q,page,sort)
#### q
1. 用來接收「搜尋關鍵字」，如 'q' : '吹風機'
#### page
1. 回傳資料預設是 20 筆商品，如果要繼續往後抓取，需要帶入頁數參數。
#### sort
1. 'sale/dc' : 有貨優先
2. rnk/dc : 精準度
3. prc/dc : 價錢高至低
4. prc/ac : 價錢低至高
5. new/dc : 新上市


In [13]:
SEARCH_URL = "https://ecshweb.pchome.com.tw/search/v4.3/all/results"
response = requests.get(SEARCH_URL, params={'q': '吹風機', 'page': 1, 'sort': 'sale/dc'}, timeout=10)
print(response.status_code)
response.url

200


'https://ecshweb.pchome.com.tw/search/v4.3/all/results?q=%E5%90%B9%E9%A2%A8%E6%A9%9F&page=1&sort=sale%2Fdc'

# 當使用者輸入隨便一個關鍵字以及指定頁數時，都能夠請求到

In [14]:
import requests
SEARCH_URL = "https://ecshweb.pchome.com.tw/search/v4.3/all/results"
def pchome (keyword : str , inquiry_page : int = 1  , sort :str = 'sale/dc'):
    search_keyword = keyword.strip()
    Inquiry_keyword = search_keyword

    response = requests.get(SEARCH_URL, params={'q': Inquiry_keyword,
                                            'page': inquiry_page,
                                            'sort': sort
                                            }, timeout=10)
    return response.json()
if __name__ == "__main__":
  inquiry_keyword = input("搜尋關鍵字：")
  inquiry_page = input("輸入頁數：")
  data = pchome(inquiry_keyword,inquiry_page)
  print('總共幾筆:', data['TotalRows'])
  print('這一頁幾筆:', len(data['Prods']))
  print(f'第 {inquiry_page} 頁第 1 筆資訊')
  first = data['Prods'][:5]
  # print(first)


總共幾筆: 8957
這一頁幾筆: 20
第 1 頁第 1 筆資訊
[{'Id': 'QAAP5N-A9008Y5ZN', 'PCateId': ['105026004001000'], 'PicS': '/items/QAAP5NA9008Y5ZN/000001_1524039252.jpg', 'PicB': '/items/QAAP5NA9008Y5ZN/000001_1524039252.jpg', 'Name': 'SONY 索尼 手機通話耳罩式耳麥MDR-ZX110AP送百元耳機', 'Describe': 'SONY手機通話耳罩式耳麥MDR-ZX110AP送百元耳機\\r\\n買就送超值小耳機\r\n簡約色彩任選擇 \r\n30mm高音質驅動單元 \r\n向內折疊結構設計 \r\n鋸齒狀導線，不易纏繞\r\n手機可通話設計', 'Price': 999, 'OriginPrice': 999, 'Author': '', 'Brand': '', 'PublishDate': '', 'isPChome': 1, 'isNC17': 0, 'CouponActId': ['C081010', 'C081009', 'C080918', 'C081007', 'C080502', 'C079914', 'C080194', 'C080191', 'C080914', 'C080481', 'C080482', 'C080920', 'C080476', 'C080487', 'C080185', 'C080193', 'C080498', 'C080198', 'C080442', 'C080488', 'C080504', 'C080184', 'C080195', 'C080905', 'C080196', 'C080478', 'C079297', 'C079172', 'C080186', 'C080190', 'C080448', 'C080479', 'C080201', 'C080503', 'C080484', 'C080495', 'C080501', 'C080493', 'C080780', 'C080443', 'C080485', 'C080446', 'C080915', 'C080445', 'C080919', 'C080

# 挑選需要的參數
1. 品名 = Name
2. OriginPrice = 原價
3. Price = 售價
4. PicS = 圖片
5. id =  商品網址
# 商品與圖片網址前綴 + id 與 PicS 才是完整網址
1. 圖片網址 = https://d.ecimg.tw
2. 商品網址 = https://24h.pchome.com.tw/prod/

In [16]:
print('品名:', first['Name'])
print('售價:', first['Price'])
print('原價:', first['OriginPrice'])
print('商品頁:', 'https://24h.pchome.com.tw/prod/' + first['Id'])
print('圖片:', 'https://d.ecimg.tw' + first['PicS'])


TypeError: list indices must be integers or slices, not str

# 問題 : results提取的商品原價和售價相同

In [17]:
for p in data['Prods'][:5]:
    print(f"品名 : {p['Name']}\n "
          f"原價 : {p['Price']} \n "
          f"售價 : {p['OriginPrice']} \n "
          f"商品頁 : https://24h.pchome.com.tw/prod/{p['Id']} \n "
          f"圖片 : https://d.ecimg.tw{p['PicS']}"
          )

品名 : SONY 索尼 手機通話耳罩式耳麥MDR-ZX110AP送百元耳機
 原價 : 999 
 售價 : 999 
 商品頁 : https://24h.pchome.com.tw/prod/QAAP5N-A9008Y5ZN 
 圖片 : https://d.ecimg.tw/items/QAAP5NA9008Y5ZN/000001_1524039252.jpg
品名 : XINMIN鑫米無線耳機Q15 藍牙耳機 智能耳機 觸控面板 降噪耳機 數顯 觸控彩屏 入耳式 真無線藍牙耳機
 原價 : 498 
 售價 : 498 
 商品頁 : https://24h.pchome.com.tw/prod/DMALG2-A900JLACC 
 圖片 : https://d.ecimg.tw/items/DMALG2A900JLACC/000001_1766742866.jpg
品名 : Apple 蘋果 Beats 360 耳罩式無線耳機-粉紅色
 原價 : 12500 
 售價 : 12500 
 商品頁 : https://24h.pchome.com.tw/prod/DYAJCA-A900KHR3D 
 圖片 : https://d.ecimg.tw/items/DYAJCAA900KHR3D/000001_1790229160.jpg
品名 : Apple 蘋果 Beats 360 耳罩式無線耳機-黑色
 原價 : 12500 
 售價 : 12500 
 商品頁 : https://24h.pchome.com.tw/prod/DYAJCA-A900KHQYG 
 圖片 : https://d.ecimg.tw/items/DYAJCAA900KHQYG/000001_1790227769.jpg
品名 : MEES Ear Clip Pro 開放式藍牙耳夾式耳機 氣傳導無線藍牙耳機 AI即時翻譯 HIFI高音質 IPX6防水 運動型耳機
 原價 : 1780 
 售價 : 1780 
 商品頁 : https://24h.pchome.com.tw/prod/DYAQFA-A900JO6AO 
 圖片 : https://d.ecimg.tw/items/DYAQFAA900JO6AO/000001_1769148020.jpg


# 在 ecapi-cdn API 中能找到商品真實的原價(M)與售價(P)

#### ecapi-cdn API網址前綴 :

https://ecapi-cdn.pchome.com.tw/ecshop/prodapi/v2/prod

# 問題 : ecapi-cdn 端點觸發防爬蟲／標頭檢查機制

##### 原因 : 由於伺服器會辨識並攔截非瀏覽器請求，未帶標頭時會觸發 PChome ecapi-cdn 的防爬蟲檢查機制

##### 結果 : 端點直接回傳 403 ，拒絕提供商品資料。

In [123]:
import requests
url = 'https://ecapi-cdn.pchome.com.tw/ecshop/prodapi/v2/prod'

response = requests.get(url, params={ 'id' :'DMALG2-A900IUKJS','fields':'Id,Name,Price'},timeout=10)
print(response.status_code)
response.url

403


'https://ecapi-cdn.pchome.com.tw/ecshop/prodapi/v2/prod?id=DMALG2-A900IUKJS&fields=Id%2CName%2CPrice'

# User-Agent 模擬一般瀏覽器身分
##### 用途 : 模擬一般瀏覽器身分使伺服器在以為是一般電腦瀏覽器在正常存取，而不是 Python 爬蟲腳本

##### 方法1 : 使用自己瀏覽器的 User-Agent(UA)
##### 方法 2 : 隨機產生User-Agent(UA)
參考資料 : https://ithelp.ithome.com.tw/articles/10209356 User-Agent 說明與使用

In [88]:
!pip install fake-useragent


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [131]:
from fake_useragent import UserAgent
import requests


In [132]:
 ua.random # 隨機產生 UserAgent

'Mozilla/5.0 (iPhone; CPU iPhone OS 18_1_1 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/18.1.1 Mobile/15E148 Safari/604.1'

In [3]:
url = 'https://ecapi-cdn.pchome.com.tw/ecshop/prodapi/v2/prod'
ua = UserAgent()
headers = {
            "User-Agent": ua.random,
    }# 使用隨機產生UA

response = requests.get(url, params={ 'id' :'DYBA4R-A900JJSBB','fields':'Id,Name,Price'},headers=headers,timeout=10)
print(response.status_code)
response.url
data = response.json()
data.keys()


200


dict_keys(['DYBA4R-A900JJSBB-000'])

#### 當透過商品 id 得到 value時，能發現真實的原價(M)與優惠(P)

In [4]:
first = 'DYBA4R-A900JJSBB-000'
item = data[first]
item

{'Id': 'DYBA4R-A900JJSBB-000',
 'Name': 'National Geographic 國家地理 EarBuds mini 真無線耳機 藍牙耳機 超輕雙耳 8g - 黑色',
 'Price': {'M': 1880, 'P': 1068, 'Low': None, 'DiscountText': '', 'Prime': ''}}

# 下面測試是透過results提取的商品ID，傳到ecapi-cdn取得原價與售價
### 問題 :

#### 1.發現在ecapi-cdn部分商品他輸出的原價( M )值等於0

#### 2. ecapi-cdn中部份商品是沒有M與P的資料

#### 2. 有些限時優惠，反而是在results中才會有


In [9]:
from fake_useragent import UserAgent
import requests

SEARCH_URL = "https://ecshweb.pchome.com.tw/search/v4.3/all/results"
PROD_URL = "https://ecapi-cdn.pchome.com.tw/ecshop/prodapi/v2/prod"
IMG_BASE_URL = "https://cs-a.ecimg.tw"


def build_product_url(prod_id: str) -> str:
  return f"https://24h.pchome.com.tw/prod/{prod_id}"


def build_image_url(pic_info: dict) -> str | None:
  if not isinstance(pic_info, dict):
    return None
  img_path = pic_info.get("B") or pic_info.get("S")
  if not img_path:
    return None
  if str(img_path).startswith("http"):
    return str(img_path)
  return f"{IMG_BASE_URL}{img_path}"


def get_PROD_by_Price(
    keyword: str,
    inquiry_page: int = 1,
    sort: str = "sale/dc",
    limit: int = 12,
):
  search_keyword = keyword.strip()
  if not search_keyword:
    return []

  ua = UserAgent()
  headers = {
      "User-Agent": ua.random,
      "Referer": "https://24h.pchome.com.tw/",
  }

  # 搜尋端點先多抓一些候選 ID，避免過濾掉非關鍵字商品後不足 limit 筆
  fetch_count = max(limit * 2, 24)
  search_response = requests.get(
      SEARCH_URL,
      params={
          "q": search_keyword,
          "page": inquiry_page,
          "pageCount": fetch_count,
          "sort": sort,
      },
      headers=headers,
      timeout=10,
  )
  search_response.raise_for_status()
  search_data = search_response.json()

  prods = search_data.get("Prods", [])
  prods_id = [prod.get("Id") for prod in prods if prod.get("Id")][:fetch_count]
  if not prods_id:
    return []

  prod_response = requests.get(
      PROD_URL,
      params={
          "id": ",".join(prods_id),
          "fields": "Id,Name,Price,Pic",
      },
      headers=headers,
      timeout=10,
  )
  prod_response.raise_for_status()
  prod_data = prod_response.json()

  results = []
  kw_lower = search_keyword.lower()

  for pid in prods_id:
    info = prod_data.get(pid) or next(
        (
            v
            for k, v in prod_data.items()
            if str(k).startswith(pid) and isinstance(v, dict)
        ),
        None,
    )
    if not isinstance(info, dict):
      continue

    prod_name = info.get("Name") or ""
    # 只保留品名中包含關鍵字的商品
    if kw_lower not in prod_name.lower():
      continue

    price_info = (
        info.get("Price", {}) if isinstance(info.get("Price"), dict) else {}
    )
    pic_info = info.get("Pic", {}) if isinstance(info.get("Pic"), dict) else {}
    actual_id = info.get("Id") or pid

    results.append({
        "Id": actual_id,
        "Name": prod_name,
        "M": price_info.get("M"),
        "P": price_info.get("P"),
        "Url": build_product_url(actual_id),
        "Image": build_image_url(pic_info),
    })

    if len(results) >= limit:
      break

  return results


if __name__ == "__main__":
  inquiry_keyword = input("搜尋關鍵字：").strip()
  raw_page = input("輸入頁數（預設 1）：").strip()
  inquiry_page = int(raw_page) if raw_page.isdigit() else 1

  items = get_PROD_by_Price(
      keyword=inquiry_keyword,
      inquiry_page=inquiry_page,
      limit=12,
  )

  print(f"共取得 {len(items)} 筆符合「{inquiry_keyword}」的商品：")
  for idx, item in enumerate(items, start=1):
    print(
        f"{idx}. 品名: {item['Name']} | M: {item['M']} | P: {item['P']}\n"
        f"   網址: {item['Url']}\n"
        f"   圖片: {item['Image']}"
    )

共取得 12 筆符合「耳機」的商品：
1. 品名: SONY手機通話耳罩式耳麥MDR-ZX110AP送超值小耳機 | M: 0 | P: 999
   網址: https://24h.pchome.com.tw/prod/QAAP5N-A9008Y5ZN-000
   圖片: https://cs-a.ecimg.tw/items/QAAP5NA9008Y5ZN/000001_1524039252.jpg
2. 品名: XINMIN鑫米無線耳機Q15 藍牙耳機 智能耳機 觸控面板 降噪耳機 數顯 觸控彩屏 入耳式 真無線藍牙耳機 | M: 999 | P: 498
   網址: https://24h.pchome.com.tw/prod/DMALG2-A900JLACC-000
   圖片: https://cs-a.ecimg.tw/items/DMALG2A900JLACC/000001_1766742866.jpg
3. 品名: Beats 360 耳罩式無線耳機-粉紅色 | M: 12500 | P: 12500
   網址: https://24h.pchome.com.tw/prod/DYAJCA-A900KHR3D-000
   圖片: https://cs-a.ecimg.tw/items/DYAJCAA900KHR3D/000001_1790229160.jpg
4. 品名: Beats 360 耳罩式無線耳機-黑色 | M: 12500 | P: 12500
   網址: https://24h.pchome.com.tw/prod/DYAJCA-A900KHQYG-000
   圖片: https://cs-a.ecimg.tw/items/DYAJCAA900KHQYG/000001_1790227769.jpg
5. 品名: MEES Ear Clip Pro 耳夾式氣傳導無線藍牙耳機 | M: 3199 | P: 1780
   網址: https://24h.pchome.com.tw/prod/DYAQFA-A900JO6AO-000
   圖片: https://cs-a.ecimg.tw/items/DYAQFAA900JO6AO/000001_1769148020.jpg
6. 品名: 【PHILIPS 飛利浦】AI智能全觸控

# 一、載入函式庫與套件
1. **import json :** 將 JSON 格式字串解析成 Python 字典或清單
2. **from pathlib import Path :**  處理資料夾與檔案路徑
3. **import random :** 讓連續抓資料時每次休息的秒數不一樣，避免因為固定同時間而被封鎖。
4. **import re :**  用規則抓出括號裡的內容，把外層函式包裝去掉，只留下裡面的 JSON 字串。
5. **import time :**  在連續請求之間做短暫停頓，避免短時間內打太多 API 請求
6. **import pandas as pd :** 把抓到的商品轉成 DataFrame、將價格轉為數值（pd.to_numeric）、檢查空值（pd.notna）以及計算折扣百分比。
7. **import requests :**  向 PChome 搜尋列表 API

In [ ]:
import json  # 將 JSON 格式字串解析成 Python 字典或清單
from pathlib import Path # 處理資料夾與檔案路徑
import random #讓連續抓資料時每次休息的秒數不一樣，避免因為固定同時間而被封鎖。
import re # 用規則抓出括號裡的內容，把外層函式包裝去掉，只留下裡面的 JSON 字串。
import time # 在連續請求之間做短暫停頓，避免短時間內打太多 API 請求
import pandas as pd #把抓到的商品轉成 DataFrame、將價格轉為數值（pd.to_numeric）、檢查空值（pd.notna）以及計算折扣百分比。
import requests # 向 PChome 搜尋列表 API
from fake_useragent import UserAgent # 模擬一般瀏覽器身分

# 二、之後會用到的 網址前綴

In [178]:
SEARCH_URL = "https://ecshweb.pchome.com.tw/search/v4.3/all/results"  # 搜尋列表 API
PROD_API_URL = "https://ecapi-cdn.pchome.com.tw/ecshop/prodapi/v2/prod"  # 批次價格 API（Price.M=原價, Price.P=促銷價）
PRODUCT_URL = "https://24h.pchome.com.tw/prod/"  # 產品 Base URL
PHOTO_BASE_URL = "https://cs-a.ecimg.tw"  # 商品圖片 Base URL
PROD_FIELDS = "Id,Name,Price,Discount,Pic" # 為ecapi-cdn API 的 fields的參數值

# 三、模擬一般瀏覽器身分

In [179]:
ua = UserAgent()
HEADERS = {
    "User-Agent": ua.random, # 隨機產生ua
}

# 將商品圖片整理成完整的圖片網址

In [188]:
def _build_photo_url(prod) :
  # 組合 PChome 商品圖片網址（使用 PicB 大圖）
  raw_pic = str(prod.get("PicB"))
  if not raw_pic:
    return "沒有圖片"
  return f"{PHOTO_BASE_URL}/{raw_pic}"

# 將ecapi-cdn API 單筆商品資料裡抓取M和P

In [ ]:
def M_and_P_ecapi(info: dict) :
  #從 ecapi-cdn 商品資料中取得 M（原價）與 P（促銷價）
  price_block = info.get("Price")  #M和P都在Price字典裡，因此要先從Price提取
  if isinstance(price_block, dict):   # isinstance(物件, 類別) 用於判斷特定例外實例是否為該類別（或其父類別）的子代 : 3.Python的套件 ->錯誤與例外處理方式->常用語法與自定義例外
    raw_m = price_block.get("M")
    raw_p = price_block.get("P")
  else:
    raw_m = info.get("M")
    raw_p = info.get("P")

# 如果在 ecapi-cdn 商品資料中沒有取得我要的M和P時，透過pd.to_numeric() 來強制將資料轉換為數字，並且將無法正確轉換的資料轉換為 NaN，防止報錯
  m_num = pd.to_numeric(raw_m, errors="coerce")
  p_num = pd.to_numeric(raw_p, errors="coerce")
# 只把大於 0 的有效數字轉成 float 存為 orig_m；如果是 0、負數或空值就設為 None，避免後面算折扣率時除以零或出現怪數字。
  orig_m = float(m_num) if pd.notna(m_num) and float(m_num) > 0 else None  #pd.notna() 用來檢查資料是否「不是缺失值」，若是有效資料則回傳 True，若是缺失值（如 NaN）則回傳 False
  promo_p = float(p_num) if pd.notna(p_num) and float(p_num) > 0 else None
  return orig_m, promo_p

#  從ecapi-cdn API 中抓取原始資料

In [ ]:
def fetch_ecapi_mp_batch(
    prod_ids: list[str],
    batch_size: int = 35, #每次請求商品 ID，預設為 35 筆
    min_delay: float = 0.25,  # 抓取價格時間 0.25~0.6秒
    max_delay: float = 0.6,
    timeout: int = 10, # 等待伺服器回應的秒數，預設 10 秒
) :
  """批次向 ecapi-cdn 查詢多個商品 ID 的原價(M)與促銷價(P)，支援 -000 尾碼比對，回傳 {id: (M, P)}"""
  clean_ids: list[str] = []
  seen: set[str] = set()
  for pid in prod_ids:
    s = str(pid).strip()
    if s and s not in seen:
      seen.add(s)
      clean_ids.append(s)

  mp_map: dict[str, tuple[float | None, float | None]] = {}
  if not clean_ids:
    return mp_map

  for idx in range(0, len(clean_ids), batch_size):
    chunk = clean_ids[idx : idx + batch_size]
    try:
      resp = requests.get(
          PROD_API_URL,
          params={
              "id": ",".join(chunk),
              "fields": PROD_FIELDS,
          },
          headers=HEADERS,
          timeout=timeout,
      )
      resp.raise_for_status()
      data = resp.json()

      for pid in chunk:
        info = data.get(pid) or data.get(f"{pid}-000")
        if not isinstance(info, dict):
          matched_key = next(
              (k for k in data.keys() if str(k).startswith(pid)),
              None,
          )
          info = data.get(matched_key) if matched_key else None

        if not isinstance(info, dict):
          continue

        orig_m, promo_p = M_and_P_ecapi(info)
        if orig_m is not None or promo_p is not None:
          mp_map[pid] = (orig_m, promo_p)
    except Exception as e:
      print(f"批次查詢 ecapi-cdn 價格中斷：{e}")

    if idx + batch_size < len(clean_ids):
      time.sleep(random.uniform(min_delay, max_delay))

  return mp_map